# Standalone ATIF Evaluation with Custom Evaluators

This notebook demonstrates using `nvidia-nat-eval` as a **fully standalone**
evaluation component with custom evaluators — no LLM endpoints, no YAML config,
no NeMo Agent Toolkit workflow, and no evaluator plugins required.

We define two simple evaluators inline:
- **Exact Match** — binary score: does the agent output match the expected answer?
- **Tool Count** — counts how many tool calls the agent made in the trajectory

These evaluators inherit from `AtifBaseEvaluator` and implement item-level
scoring via `evaluate_atif_item`, while the base class handles concurrent
orchestration for `evaluate_atif_fn`.
They run through the `EvaluationHarness`.

**What this proves:**
- `nvidia-nat-eval` works as a standalone package — no API keys, no LLMs, no plugins
- Custom evaluators are plain Python classes with a single async method
- ATIF trajectories from any agent framework can be scored immediately

**Requirements:** `nvidia-nat-eval` only

## 1. Install Dependencies

In [ ]:
# Only nvidia-nat-eval is needed — no ragas, no langchain, no API keys.
#
# For released versions:
# !uv pip install nvidia-nat-eval
#
# Install pinned alpha release from PyPI:
!uv pip install -q "nvidia-nat-eval==1.6.0a20260309"

## 2. Define ATIF Trajectories

We create two sample trajectories that could have been produced by any agent
framework. One uses tools to answer correctly; the other answers incorrectly
without using tools.

In [ ]:
from nat.data_models.atif import ATIFTrajectory

# Sample 1: Agent uses a calculator tool and gets the right answer
trajectory_correct = ATIFTrajectory.model_validate({
    "schema_version": "ATIF-v1.6",
    "session_id": "sample-001",
    "agent": {"name": "test-agent", "version": "1.0"},
    "steps": [
        {
            "step_id": 1,
            "source": "user",
            "message": "What is 12 * 15 + 8?",
        },
        {
            "step_id": 2,
            "source": "agent",
            "message": "I'll calculate this step by step.",
            "tool_calls": [
                {
                    "tool_call_id": "call_001",
                    "function_name": "calculator__multiply",
                    "arguments": {"a": 12, "b": 15},
                }
            ],
            "observation": {
                "results": [{"source_call_id": "call_001", "content": "180"}]
            },
        },
        {
            "step_id": 3,
            "source": "agent",
            "message": "Now adding 8.",
            "tool_calls": [
                {
                    "tool_call_id": "call_002",
                    "function_name": "calculator__add",
                    "arguments": {"a": 180, "b": 8},
                }
            ],
            "observation": {
                "results": [{"source_call_id": "call_002", "content": "188"}]
            },
        },
        {
            "step_id": 4,
            "source": "agent",
            "message": "188",
        },
    ],
})

# Sample 2: Agent answers without tools and gets it wrong
trajectory_wrong = ATIFTrajectory.model_validate({
    "schema_version": "ATIF-v1.6",
    "session_id": "sample-002",
    "agent": {"name": "test-agent", "version": "1.0"},
    "steps": [
        {
            "step_id": 1,
            "source": "user",
            "message": "What is the capital of France?",
        },
        {
            "step_id": 2,
            "source": "agent",
            "message": "London",
        },
    ],
})

print(f"Trajectory 1: {len(trajectory_correct.steps)} steps")
print(f"Trajectory 2: {len(trajectory_wrong.steps)} steps")

## 3. Build `AtifEvalSample` Objects

Each sample pairs a trajectory with ground-truth expected output and the
agent's actual output.

In [ ]:
from nat.plugins.eval.evaluator.atif_evaluator import AtifEvalSample

samples = [
    AtifEvalSample(
        item_id="math-q1",
        trajectory=trajectory_correct,
        expected_output_obj="188",
        output_obj="188",
    ),
    AtifEvalSample(
        item_id="geo-q1",
        trajectory=trajectory_wrong,
        expected_output_obj="Paris",
        output_obj="London",
    ),
]

for s in samples:
    print(f"  {s.item_id}: output={s.output_obj!r}, expected={s.expected_output_obj!r}")

## 4. Define Custom Evaluators

For simpler and safer ATIF custom evaluators, use `AtifBaseEvaluator`.

It provides:
- bounded concurrency
- built-in `asyncio.gather` orchestration
- average score calculation

You only implement item-level scoring:

```python
async def evaluate_atif_item(self, sample: AtifEvalSample) -> EvalOutputItem
```

In [ ]:
from nat.data_models.evaluator import EvalOutputItem
from nat.plugins.eval.evaluator.atif_base_evaluator import AtifBaseEvaluator
from nat.plugins.eval.evaluator.atif_evaluator import AtifEvalSample


class ExactMatchEvaluator(AtifBaseEvaluator):
    """ATIF-native evaluator: binary score based on exact output match."""

    def __init__(self, normalize_case: bool = True, max_concurrency: int = 4) -> None:
        super().__init__(max_concurrency=max_concurrency)
        self._normalize_case = normalize_case

    def _normalize(self, value: object) -> str:
        text = "" if value is None else str(value).strip()
        return text.casefold() if self._normalize_case else text

    async def evaluate_atif_item(self, sample: AtifEvalSample) -> EvalOutputItem:
        """Score one ATIF sample with exact match comparison."""
        expected = self._normalize(sample.expected_output_obj)
        generated = self._normalize(sample.output_obj)
        score = 1.0 if generated == expected else 0.0

        return EvalOutputItem(
            id=sample.item_id,
            score=score,
            reasoning={
                "comparison": "exact-match",
                "expected": expected,
                "generated": generated,
                "match": score == 1.0,
            },
        )


class ToolCountEvaluator(AtifBaseEvaluator):
    """ATIF-native evaluator: scores each sample by trajectory tool call count."""

    def __init__(self, max_concurrency: int = 4) -> None:
        super().__init__(max_concurrency=max_concurrency)

    def _count_tool_calls(self, sample: AtifEvalSample) -> tuple[int, list[str]]:
        steps = getattr(sample.trajectory, "steps", None) or []
        tool_names: list[str] = []
        for step in steps:
            for tc in getattr(step, "tool_calls", None) or []:
                tool_names.append(tc.function_name)
        return len(tool_names), tool_names

    async def evaluate_atif_item(self, sample: AtifEvalSample) -> EvalOutputItem:
        """Score one ATIF sample by trajectory tool call count."""
        count, tool_names = self._count_tool_calls(sample)

        return EvalOutputItem(
            id=sample.item_id,
            score=float(count),
            reasoning={
                "comparison": "tool-count",
                "trajectory_tool_call_count": count,
                "tool_calls": tool_names,
            },
        )


print("Evaluators defined: ExactMatchEvaluator, ToolCountEvaluator")

## 5. Run Evaluation via `EvaluationHarness`

Pass the custom evaluators directly to the harness — no builder, no registry,
no config object needed.

In [ ]:
from nat.plugins.eval.runtime.eval_harness import EvaluationHarness

harness = EvaluationHarness()

results = await harness.evaluate(
    evaluators={
        "exact_match": ExactMatchEvaluator(normalize_case=True),
        "tool_count": ToolCountEvaluator(),
    },
    atif_samples=samples,
)

print("=" * 60)
print("Evaluation Results")
print("=" * 60)
for name, output in results.items():
    print(f"\n--- {name} (avg={output.average_score}) ---")
    for item in output.eval_output_items:
        print(f"  {item.id}: score={item.score}")
        print(f"    {item.reasoning}")

## 6. Summary

| Aspect | What we used |
|---|---|
| **Package** | `nvidia-nat-eval` only |
| **LLMs** | None |
| **API keys** | None |
| **YAML config** | None |
| **NeMo Agent Toolkit workflow** | None |
| **Builder / Registry** | None |

Custom evaluators are plain Python classes that inherit from `AtifBaseEvaluator`
and implement one item-level method:
`async def evaluate_atif_item(self, sample) -> EvalOutputItem`.

`AtifBaseEvaluator` handles concurrency (`asyncio.gather` + semaphore) and
average score computation, so evaluators stay concise.

They can be passed directly to `EvaluationHarness.evaluate()` and run against
any ATIF trajectory — regardless of which agent framework produced it.

For evaluators that need LLMs (RAGAS, LLM-as-judge, etc.), see the companion
notebook `eval_atif_standalone.ipynb` which uses the builder to construct those.